# Human-in-the-loop coding prompts

These are the prompts used in the human-in-the-loop (HITL) pipeline that coded students' end-of-semester memos for authenticity and research component codes. This notebook is for reference only. It contains the prompts and the settings they were run with, but none of the pipeline code. The original pipeline called models through an institutional AI gateway so would need to be adapted.

**Example quotes are paraphrased.** The code definitions include example student sentences. The real examples come from student writing, so every one of them has been replaced with a synthetic paraphrase that keeps a similar coding signal. For the same reason, the real exemplar bank isn't included. `dummy_exemplar_bank.xlsx`, in this folder, has the same structure with a few sample rows made of fake data.

## How the prompts fit together

Each memo is coded separately for each code, in two stages:

1. **Span finder (S-agent).** It gets the full memo plus a code definition (`S_AGENT_DEFS`). It returns every character span that *could* match the code, and is instructed to maximize recall.
2. **Snap to sentences.** Each span is widened to the full sentences it overlaps, matching the sentence segmentation the human coders used.
3. **Verifier (V-agent).** It gets each candidate sentence, that code's full definition including the exclusion rules and negative examples (`V_AGENT_DEFS`), and the 4 most similar exemplars from the exemplar bank. It returns keep or reject, and is instructed to lean toward keeping.
4. **Human review.** A researcher reviews every kept sentence in the context of the whole memo and makes the final call. **The LLM output is a set of suggestions, and every final code is a human decision.**

The exemplars are retrieved by cosine similarity of `all-MiniLM-L6-v2` sentence embeddings, from three pools:
- **direct:** exemplars for this code (*Typical*, *Close but yes*, *Close but no*)
- **co_code:** exemplars of other codes whose `co_code` column lists this code
- **boundary:** *Close but yes* / *Close but no* exemplars of other codes whose `boundary_code` column lists this code

`V_AGENT_USER` below shows how each pool is worded in the prompt.

| short code | code |
|---|---|
| DRR | Did real research |
| FLS | Felt like a scientist |
| URR | Understanding real research |
| S3 | Failure |
| S5 | Iteration |
| S6 | Scientific practices |
| S7 | Decision making |

DRR, FLS and URR together make up **direct authenticity (DA)**.

## Settings used

The settings the prompts were run with. The prompt functions below read `USE_EXEMPLAR_BANK` and `CONTEXT_SENTENCES`.

In [ ]:
S_MODEL           = "gpt-4.1-mini"   # span finder; cheaper model was used since precision was not the goal
V_MODEL           = "kimi-k2.5"      # verifier (thinking disabled); a bigger model to help precision
TEMPERATURE       = 0                # both agents, one call per (memo, code)
USE_EXEMPLAR_BANK = True             # verifier sees retrieved exemplars
EXEMPLAR_TOP_K    = 4                # exemplars per verifier call
CONTEXT_SENTENCES = 0                # neighbouring sentences shown to the verifier
EMBEDDING_MODEL   = "all-MiniLM-L6-v2"   # exemplar retrieval

## 1. Span-finder (S-agent) prompts

`S_AGENT_DEFS` holds the inclusion-oriented definition of each code. `S_AGENT_SYSTEM` and `S_AGENT_USER` build the system and user messages.

In [ ]:
import textwrap

# ================================================================
# 1. SPAN AGENTS (DRR, FLS, URR, S3, S5, S6, S7)
# ================================================================

S_AGENT_DEFS = {

    "DRR": textwrap.dedent("""
        DRR (Did real research)

        MUST INCLUDE:
        - Student explicitly states that the work they did in the class *was* real research or real science or that the data they collected/used was real
        - The student conveys — explicitly OR through clear evaluative/reflective framing — that the work they did was real/authentic (an authenticity claim, not mere experiment narration). A strong evaluative signal counts even if the exact words "real research" are not used.

        INCLUDE WHEN:
        - Student says they did, conducted, or carried out real research or actual science.
        - Student uses words like "real", "actual", or "real-world" to describe the research or data they produced.
        - Student indicates that THEY believe their outputs or products of their work (data, results, experiments) were genuine research artifacts
        - Student frames the work as their OWN research project, experiment, or question that they chose or designed.
        - Student conveys that this is how real science actually works, or that they figured things out from scratch / did open-ended real investigation rather than following a known-answer procedure.
        - Advice or future-tense versions of any of the above aimed at future students.
        - The class being centered on real / actual experiments or real data, or advice urging future students to engage with how physics / science is actually DONE (the process of doing science, not just its conclusions), counts.

        EXAMPLES (Positive):
        - "There is a real difference between a canned lab and getting to chase a phenomenon you actually picked yourself."
        - "The final project is where it clicks: you are designing the experiment, not just running someone else's procedure."
        - "It is one of the few classes where the data is truly yours, with all the mess and ownership that comes with it."
        - "You learn experimental physics by actually doing an experiment of your own, not by reading about one."

        SPAN GUIDANCE:
        - Select spans that include enough surrounding context to capture what specifically about the work was real or authentic.
        - Avoid selecting isolated phrases like "actual data" or "real research" without the rest of the sentence.
        - Prefer full sentences even if the sentence contains some extra irrelevant material.

        CO-OCCURRENCE POLICY:
        - If a span meets the criteria for this code, assign it even if it may also qualify for other codes.
        - Do not attempt to resolve overlap.
    """).strip(),

    "FLS": textwrap.dedent("""
        FLS (Felt like a scientist)

        MUST INCLUDE (at least one of):
        - Student relates themselves, their work as a scientist/researcher/engineer, or their way of thinking to the identity or activities of a real scientist, researcher, or physicist; DO NOT INFER that they felt like a scientist based on the activities they did in lab
        - Student explicitly discusses feeling that the skills they learned are applicable to their future or their work as a scientist/researcher/engineer; DO NOT INFER that the skills will be applicable to their future work based on the activities they did in lab

        INCLUDE WHEN:
        - Student says they felt like a scientist, researcher, or physicist while doing the work.
        - Student discusses feeling that the skills they learned are applicable to their future or their work as a scientist/researcher/engineer.
        - Student frames themselves in the role of a scientist/researcher/engineer (e.g., "thinking like a scientist", "working like a researcher").
        - Student associates science as the work of human researchers or scientists and references understanding what the researchers themselves do
        - Explicit identity statements addressed to the reader still count — e.g., "you’re now a ‘researcher,’" "you'll become the ‘expert’ here," "this is what thinking creatively like a scientist looks like." Do not skip these because they are phrased as advice to a future student; the projection of the scientist/researcher identity is the signal. The salutation-only exclusion (X1) does NOT apply when the identity word carries a substantive role claim rather than just opening the memo.

        EXCLUDE WHEN:
        - X1: The sentence is a memo header or salutation (e.g., "Dear Future Researchers,", "Memo to Future Researchers", "To a future PHYS 1110 researcher:", "Dear future physicists,") — these contain the word "researcher" but carry no affective or identity content from the student.

        EXAMPLES (Positive):
        - "This class was a solid first look at how scientists think through and analyze experiments."
        - "I hope this lab helps you grow as a researcher the way it helped me."
        - "Their aim should be to learn something, whether the hypothesis holds up or surprises push them to grow, picking up lessons from mistakes as an experimental scientist would." (advice-form identity projection — "as an experimental scientist" signals the student believes the class cultivates that identity)
        - "Get excited, because in this class you see how researchers come up with new ideas about how the world works."
        - "Rather than memorizing laws, you become the ‘expert’ who shows where they come from — that is what creative scientific thinking looks like." (projects the researcher/expert identity onto the student even in advice form)
        - "What you learn here is genuinely useful, particularly if you plan to do research later on."

        SPAN GUIDANCE:
        - Select spans that include enough surrounding context to show the identity or feeling dimension.
        - Avoid selecting isolated phrases like "scientist" or "researcher" without the rest of the sentence.
        - Prefer full sentences even if the sentence contains some extra irrelevant material.

        CO-OCCURRENCE POLICY:
        - If a span meets the criteria for this code, assign it even if it may also qualify for other codes.
        - Do not attempt to resolve overlap.
    """).strip(),

    "URR": textwrap.dedent("""
URR (Understanding real research)

MUST INCLUDE:
- Clear explicit evidence in the words of the student that they gained a better understanding of what science, research, or experimental physics is like.
- Prototypical URR phrasings to ALWAYS propose: "taught me how real science/research works," "the actual process of designing/doing experiments," "a better understanding of experimental physics," "learned what research is really like." Propose these even when brief, embedded in a skills list, or surrounded by experiment narration — do NOT skip a clearly URR sentence just because the rest of the memo is about specific activities.
- Students MUST strongly indicate or state explicitly that THEY BELIEVE they learned about how the process of research works; DO NOT INFER that they understand real research based on the activities they did in the lab

INCLUDE WHEN:
- Student describes gaining an understanding of what it is like to do research or what research involves.
- Student says they have a better understanding of conducting experiments in a broader research or scientific context than the classroom
- PROCESS is key. Emphasizes elements of the research process or features of science/physics.  Can include "learning how to do science/research/physics" as well as "learning how real science/research/physics is done" and "learning what researchers/scientists do"
- Student is talking about general features of physics/experiment that they have learned, such as understanding that data can be messy or that it requires patience
- Student describes collaboration, teamwork, or peer exchange as part of how research/science actually works (e.g., "collaborating with classmates is a big part of how experimental physics gets done", "big discoveries almost never come from a single person working alone")
- Student describes the course itself as being about the scientific/research process (e.g., "the class focuses less on physics content and more on the scientific process of running good experiments")
- A reference to the discipline itself — "experimental physics," "scientific experiments," "experimentation," or "research" as a field — counts EVEN WHEN it appears inside a list of specific skills or class tasks. Do not skip a sentence just because it also names procedures (data analysis, setting up experiments, statistical analysis); if any part frames learning/understanding about experimental physics or scientific experiments as a discipline (e.g., "develop your experimental physics knowledge," "learn to correctly design and run scientific experiments"), propose it.
- Guest lectures / speakers: when the student references a guest researcher, speaker, or lecture that gave them a better idea of the research going on at the school or what real research is like, propose it as URR (e.g., "a visiting astronomer talked to us about what their research is actually like," "definitely go to the guest talks").

EXAMPLES (Positive):
- "This lab taught me a great deal about experimental physics and the tools you need to run experiments."
- "The class was hard but worth it; beyond methods and data analysis, I learned a lot about teamwork, communication, and solving problems in a research context."
- "I learned how to hold physics models up against real-world experiments and connect the different concepts we tested."
- "This class showed me how real science works — not formula plugging like in lecture, but actually designing experiments."
- "I started seeing experimental physics as a subject in its own right and got a clearer picture of what research actually involves."

SPAN GUIDANCE:
- Select spans that include enough surrounding context to show what the student now understands about research.
- Avoid selecting isolated phrases like "understanding" or "experimental physics" without the rest of the sentence.
- Prefer full sentences even if the sentence contains some extra irrelevant material.

CO-OCCURRENCE POLICY:
- If a span meets the criteria for this code, assign it even if it may also qualify for other codes.
- Do not attempt to resolve overlap.
    """).strip(),


    "S3": textwrap.dedent("""
        S3 (Failure)
        MANY KEYWORDS FOR THIS CODE ARE IN QUOTES. IF YOU SEE SUCH A KEYWORD OR ITS NOUN/VERB/ADJECTIVE/ADVERB VERSION, CODE IT!!!

        MUST INCLUDE (at least one of):
        - "Difficulties' experienced during the lab such as things going wrong, struggling to get something to work, or major issues encountered that forced students to "redo' work or change course
        - General observations about the non-ideal or "messy" nature of data/experimental work (mentions of "mistakes", "problems", "difficulties", "struggles', "failures", or implied failures like "troubleshooting" or "adaptability" count)
        - Student observes a need for patience or experiments tending to take longer than expected

        INCLUDE WHEN:
        - ***Student uses words like "trial and error", "troubleshoot", "debug," "fix," or other words (even if they have positive connotation) that imply (possibly previous) problems or failure to describe their lab experience
        - ***Noting a shift in thought from the purpose of labs as confirming theory or getting a correct answer to *working with the data acquired* (implies data is "messy", "imperfect", etc.)
        - ***Student describes anything being generally or specifically difficult or challenging, or have issues with getting an experiment to work
        - Student emphasizes "flexibility", "adaptability", or "dealing with the data [they] get"
        - Student notes or implies that data is "messy", uncertain, not ideal, or difficult to fit well
        - Student discusses something that seemed like a rather large "catastrophe" for their group that likely caused major distress
        - Soft-negative descriptions of method or data failure also count — e.g., "inability to X", "was not great", "did not work well", "couldn't get Y to work", "limited by", "confounded by"
        - Idiomatic or reflective summaries of failure experiences also count — e.g., "trials and tribulations", "mistake", broad statements that experimenting is "messy" or not about getting a "right" answer

        EXAMPLES (Positive):
        - "It was frustrating early on, but learning to make sense of messy, imperfect data ended up being one of the best parts of the class."
        - "That said, our experiment wasn't perfect; it had some flaws and errors."
        - "One big lesson we learned: a hypothesis that turns out wrong doesn't mean the experiment failed."
        - "Because we couldn't get a trigonometric fit to work, we linearized the data and found an exponential fit matched well."
        - "Pooling the data hurt our analysis, since each detector had a different threshold voltage and was tested at a different time, which introduced confounding factors."

        SPAN GUIDANCE:
        - Select spans that include enough surrounding context to show what aspect of their experience the student associated with failure
        - Avoid selecting isolated phrases like “fail” or "troubleshoot" without the rest of the sentence.
        - Prefer full sentences even if the sentence contains some extra irrelevant material.

        CO-OCCURRENCE POLICY:
        - If a span meets the criteria for this code, assign it even if it may also qualify for other codes.
        - Do not attempt to resolve overlap.
    """).strip(),


    "S5": textwrap.dedent("""
        S5 (Iteration)

        MUST INCLUDE (at least one of):
        - Statements about reducing error in, refining, or improving data/uncertainty/experiments in general
        - How repeating certain activities in their experiment improved it
        - Changing approach to their experiment or doing another experiment built on what they had started
        - General observations that iteration or modificaiton is important in experimentation

        INCLUDE WHEN:
        - Terms like "repetition", "iteration", "redo", "refine", "change", "improve", “trial and error”, "second/multiple/more/numerous trials/experiments", "adjust", "troubleshoot", "recalibrate", "modify", "rebuild", "adapt", "redesign", "alter," or other words indicating change and repetition are used.
          - These terms can be in noun, verb, adjective, adverb, etc. form.
        - Student described learning about the importance of making improvements and/or repeated trials
        - Discussion of activities done to reduce uncertainty or minimize error
        - Student talks about designing a new experiment after a different one failed
        - Student describes learning from a mistake or failure and applying that insight to improve their approach
        - If in the specific context of a student's experiment, must indicate that improvement is being made to the experiment, not just that multiple trials were conducted
        - Future-student or advice-form framing of iteration also counts (e.g., "a new student should consider factors and try to reduce them", "the main goal should be making the model more accurate", "you should refine your approach")
        - A method, model, or fit was tried, found wanting, and replaced by another — even when the failure is in observational/passive voice ("inability to fit X, so we used Y" or "after linearizing, an exponential fit matched well")
        - Outcome-framed refinement counts: when refining/revising/improving the procedure is stated as the *cause* of a result (e.g., \"because we refined our procedure, our data became more reliable\"), the refinement clause is S5 even though it is phrased as leading to an outcome.

        EXAMPLES (Positive):
        - "In each lab we designed our own procedure, wrote it up carefully, and then improved it based on what didn't work."
        - "Getting this setup working was hard and took us several rounds of changes."
        - "The class emphasizes designing experiments, trial and error, and making sense of uncertainty."
        - "New students should think about what could affect their results and work to minimize those factors so they can answer their question carefully."
        - "Because we couldn't get a trigonometric fit to work, we linearized the data and found an exponential fit matched well."

        SPAN GUIDANCE:
        - Select spans that include enough surrounding context to show the student did iteration to refine their experiment
        - Avoid selecting isolated phrases like “refine” or "iteration" without the rest of the sentence.
        - Prefer full sentences even if the sentence contains some extra irrelevant material.

        CO-OCCURRENCE POLICY:
        - If a span meets the criteria for this code, assign it even if it may also qualify for other codes.
        - Do not attempt to resolve overlap.
    """).strip(),

    "S6": textwrap.dedent("""
        S6 (Scientific practices)

        *IF A SPAN HAS ONE OR MORE OF THESE KEYWORDS, NOTE THE SPAN AS AN S6 CANDIDATE IMMEDIATELY*: teamwork/team, coding, experimenting, data analysis (statistics, statistical test, model, fit), design, and presenting.
        If any of these phrases or related words/concepts like critical thinking are mentioned or implied--even vaguely or within the context of one experiment/advice to future students without any further reflection--they should still be coded as S6.

        MUST INCLUDE:
        - Statements indicating/implying that the student DID, LEARNED TO DO, IMPLIES, or NAMES specific scientific practices in the context of THIS class or their final project
        - First- or second-person singular/plural framing, or third person framing related to the lab/class ("I learned…", "You will use…", "we calculated/fit/analyzed…", "we measured/tested…", "This lab helps develop skills in…") naming a practice or skill is sufficient — explicit reflection or evaluation language is NOT required
        - Bare project-topic descriptions ("our project investigated X") still do NOT qualify, but listing or naming the practices used does. Descriptions of experimental setups and procedures are S6.

        INCLUDE WHEN:
        - There is any mention of designing or carrying out experiments (including listing experimental steps or describing the experimental setup), analyzing/calculating data/fits, using various practices to reduce error, or collaborating with others
        - The student REFLECTS, names, or implies specific scientific practices they engaged in or learned as part of their experience (e.g., coding (in Python), data analysis, statistics, finding conclusions, any sort of teamwork, experimental design, experimentation, critical thinking, statistical testing, graphing, presenting work)
          - If any of these words are directly mentioned ("coded", "analyzed", "teamwork", etc.) the span is S6. Even if they are implied (e.g. "we fit a model" = data analysis, "we realized that our data showed" = critical thinking + finding conclusions), the span can still be S6!
        - The student discusses scientific practices used in their project with reflective or evaluative framing (e.g. we ran multiple trials to reduce error) AND/OR narrates experimental steps
        - Any mention of learning how to code or analyze data, plot, keep notes and/or keep data organized
        - The student notes a term related to a scientific practice by saying that the course (PHYS 1110) taught them this skill or that they learned it in lecture/lab
        - Advice to future students mentioning practices learned/engaged in, challenges with those practices, or hoping that THEY learn scientific practices still counts
        - Concrete advice that names a scientific practice or research habit also counts even without explicit "learn" framing (e.g., "write everything down in your notebook", "repeat trials for reliability", "check your assumptions", "judge whether results make physical sense", "assign team roles", "talk with your classmates")
        - Naming specific data-analytic practices counts even in narrative voice (e.g., fitting models to data, calculating things, computing chi-squared or residuals, linearizing data, comparing fit quality, choosing among candidate models, making conclusions from models)

        EXAMPLES (Positive):
        - "The skills I built most in this class were communication, teamwork, and persistence."
        - "To answer it, we collected and cleaned our data and then fit several models in Colab."
        - "The main things I learned were how to design experiments and how to analyze what they produce."
        - "Fitting a line to the raw data gave a large chi-squared, so the fit was poor. Once we linearized against √m, the chi-squared dropped."
        - "Record everything in your lab notebook, even details that seem unimportant — you'll be glad you did."

        SPAN GUIDANCE:
        - Select spans that include enough surrounding context to show the specific scientific practices students engaged in
        - Avoid selecting isolated phrases like "data collection", "coding", "teamwork", etc.) without the rest of the sentence.
        - Prefer full sentences even if the sentence contains some extra irrelevant material.
        - This code is more common than the others and will generally be applied most often

        CO-OCCURRENCE POLICY:
        - If a span meets the criteria for this code, assign it even if it may also qualify for other codes.
        - Do not attempt to resolve overlap.
    """).strip(),

    "S7": textwrap.dedent("""
        S7 (Decision making)

        MUST INCLUDE (at least one of):
        - Indication that students felt independent or able to make their own decisions in the lab
        - Indication that students could get creative or curious about their approaches/solutions

        INCLUDE WHEN:
        - There is a sense of agency associated with the statement.
        - Students indicate they had autonomy of choice (e.g., choosing a topic/question, deciding how to answer a research question)
        - Students express that they were able to follow their interests in the lab
        - Students say that they were able to exercise creativity in their work
        - Advice to future students that DESCRIBES the autonomy, creativity, or choice they will have in the lab/project (e.g., \"you'll come up with your own experiment\", \"you'll be able to choose your topic\", \"you'll have the freedom to explore something you're curious about\", \"you can try obscure things\") — advice form does NOT preclude S7 when the substance is about agency/autonomy
        - Setting the project's goal, topic, or research question, or advising future students to (e.g., "think early about what you're trying to learn or prove", "settle on a topic before you start", "decide what your project should be")
        - Organizing the team — choosing roles, partners, or how to delegate (e.g., "be open to who you work with", "choose suitable roles in your group", "how to delegate tasks fairly")
        - Choosing a project around your OWN INTERESTS, or coming up with / designing your own experiment (e.g., "anything that interests you", "a topic you're curious about", "come up with a unique experiment", "design your own experiment", "you were able to test anything you wanted")
        - Asserting your own ideas or intellectual independence — self-advocacy, sharing your voice, or deciding to find answers yourself rather than taking things at face value (e.g., "advocate for yourself and your ideas", "share your voice", "search for answers yourself")

        EXAMPLES (Positive):
        - \"New students should aim to design an experiment they actually care about, since that keeps them more engaged with it.\"
        - \"At first I was nervous about having to come up with my own ideas and procedure.\"
        - \"This lab showed me how fun experimenting can be when you get to be as creative as you want.\"

        SPAN GUIDANCE:
        - Select spans that include enough surrounding context to show what aspects of the lab students are making decisions about
        - Avoid selecting isolated phrases like \"choose\" or \"interesting\" without the rest of the sentence.
        - Prefer full sentences even if the sentence contains some extra irrelevant material.

        CO-OCCURRENCE POLICY:
        - If a span meets the criteria for this code, assign it even if it may also qualify for other codes.
        - Do not attempt to resolve overlap.
    """).strip(),
}



# Operational rules: maximum recall — verifier handles all exclusion.
_S_AGENT_RULES = """- Work ONLY on code {s_code}. Ignore all other themes.
- A "span" is a contiguous substring of the memo text that plausibly relates to {s_code}.
- Spans must be *verbatim* substrings of the memo, using 0-based [char_start, char_end) indices.
- Return concise spans that still contain enough context to understand the meaning.
  Include the verbs and objects needed to interpret the student's experience.
- CRITICAL: Your ONLY job is candidate generation. A downstream verifier will review every span
  you return and will reject false positives. Your goal is maximum recall — include any span that
  could plausibly relate to {s_code}, even if you are uncertain or it feels borderline.
  DO NOT self-filter based on exclusion criteria. It is far worse to miss a valid span than
  to include one the verifier rejects. When in doubt, always include.
- Only return an empty list if the memo contains absolutely nothing that could plausibly relate to {s_code}."""


PRIORITY_CODES = {"DRR", "FLS", "URR"}

def S_AGENT_SYSTEM(s_code: str, desc: str) -> str:
    rules = _S_AGENT_RULES.format(s_code=s_code)
    priority_note = (
        """
PRIORITY NOTE:
- This is a high-priority code. When in doubt, include the span — it is better to
  over-code than under-code here. The verifier will filter out false positives.
"""
        if s_code in PRIORITY_CODES
        else ""
    )
    return f"""
You are a span-finding specialist for the Authenticity dimension of a codebook for student memos written about their experience in
introductory physics lab.

Your ONLY job is to scan a single student memo and extract the exact text spans
that match this specific Authenticity theme:

  CODE: {s_code}

DETAILED THEME GUIDELINES:
{desc}
{priority_note}
OPERATIONAL RULES:
{rules}

Return STRICT JSON only, with this schema:

{{
  "spans": [
    {{
      "char_start": <int>,   // inclusive, 0-based
      "char_end": <int>,     // exclusive, 0-based
      "span_text": "<verbatim substring>"
    }},
    ...
  ]
}}

No explanations or commentary.
""".strip()


def S_AGENT_USER(memo: str) -> str:
    return f"""

Below is a single memo written by a student.
Indices are 0-based over the full string.

STUDENT MEMO:
\"\"\"{memo}\"\"\"

Your task:
Extract spans that match the requested Authenticity code only.
Return ONLY a JSON object with a 'spans' array as specified in the system prompt.
""".strip()

# ================================================================

## 2. Verifier (V-agent) prompts

`V_AGENT_DEFS` holds the full definition of each code, including the exclusion rules and negative examples. `V_AGENT_SYSTEM` and `V_AGENT_USER` build the messages, and `V_AGENT_USER` formats the retrieved exemplars.

In [ ]:
# ================================================================
# 2. VERIFIER AGENT PROMPTS
# ================================================================

# V_AGENT_DEFS holds the full definition (including EXCLUDE WHEN and EXAMPLES (Negative))
# that the V-agent uses for boundary precision. S_AGENT_DEFS (defined above) holds only
# the inclusion guidance — what the S-agent sees. The two dicts are independent;
# edits to one do not affect the other.
V_AGENT_DEFS = {

    "DRR": textwrap.dedent("""
        DRR (Did real research)

        KEY DISTINCTION: plain narration of the procedure, measurements, or results is NOT DRR.
        But code DRR whenever the sentence asserts OR clearly frames that the work — or the
        course's core activity — was real / authentic research, real data, the student's own
        research, or a genuine contribution to real research. The realness can attach to the
        nature of the course or the work, not only to a direct "the work was real" self-claim.

        MUST INCLUDE:
        - M1: Student explicitly states that the work they did in the class *was* real research or real science or that the data they collected/used was real
        - M2: The student conveys — explicitly OR through clear evaluative/reflective framing — that the work was real/authentic, that they worked with real data, or that they did their own research. A strong evaluative signal counts even without the exact words "real research"; when such a claim is present, lean toward KEEP.

        INCLUDE WHEN:
        - I1: Student says they did, conducted, or carried out real research or actual science.
        - I2: Student says that they did simulated or almost real research or science
        - I3: Student expresses that the class gave them a special chance to do their own research or gives students a chance to do their own research and that this feature is exceptional to this lab
        - I4: Student uses words like "real" or "actual" to describe the research or data they produced 
        - I5: Phrases "real data" or "actual data" are strong indications to flag as DRR 
        - I6: Student indicates that THEY believe their outputs or products of their work (data, results, experiments) were genuine experimental physics research artifacts 
        - I7: The span focused on experiencing or engaging in experimental physics
        - I8: The span conveys this is how real science actually works, or that they figured things out from scratch / did open-ended real investigation rather than following a known-answer procedure.
        - I9: Advice or future-tense versions of any INCLUDE case aimed at future students.
        - I10: The sentence frames the COURSE or the work as centered on real / actual experiments or real data, or as being (or contributing to) real research, even without an explicit "the work was real" self-claim (the class is about applying skills to actual experiments; the course work is a step toward real research).

        EXCLUDE WHEN:
        - X1: The student only describes *feeling like* a scientist/researcher without claiming the work itself was real (that is FLS — identity/role framing, not a claim about authenticity of the work).
        - X2: The student talks about *learning about* how research/science works without claiming the work itself was real (that is URR — meta-understanding of research as a process/discipline).
        - X3: The student says they "learned how to" do something without saying they actually did it (learning to do something is not doing it).
        - X4: REJECT plain narration of the final project, experiment, procedure, setup, measurements, data collection, or results — even when the work was genuinely real and used real data. Describing WHAT they did or found is NOT DRR (e.g. a sentence that only reports which variables were measured, what apparatus was used, or what result was found). Keep only when the sentence ITSELF claims the work was real / authentic or was their own research.

        EXAMPLES (Positive):
        - "There is a real difference between a canned lab and getting to chase a phenomenon you actually picked yourself."
        - "The final project is where it clicks: you are designing the experiment, not just running someone else's procedure."
        - "It is one of the few classes where the data is truly yours, with all the mess and ownership that comes with it."
        - "You learn experimental physics by actually doing an experiment of your own, not by reading about one."

        EXAMPLES (Negative):
        - "I got a lot better at analyzing data and applying physics to real-life situations over the semester." (improving a skill, not a claim the work was real research)
        - "Throughout the class I learned how to set up and do real experiments, not just the fundamentals of physics." (learned HOW to, without saying they did real research)
        - "By the end I felt comfortable forming my own research question and reasoning through how to approach it." (only saying they understand this part of the research process, that is URR)

        SPAN GUIDANCE:
        - Select spans that include enough surrounding context to capture what specifically about the work was real or authentic.
        - Avoid selecting isolated phrases like "actual data" or "real research" without the rest of the sentence.
        - Prefer full sentences even if the sentence contains some extra irrelevant material.

        CO-OCCURRENCE POLICY:
        - If a span meets the criteria for this code, assign it even if it may also qualify for other codes.
        - Do not attempt to resolve overlap.
    """).strip(),

    "FLS": textwrap.dedent("""
        FLS (Felt like a scientist)

        MUST INCLUDE (at least one of):
        - M1: Student relates themselves, their work as a scientist/researcher/engineer, or their way of thinking to the identity OR activities of a real scientist, researcher, or physicist; DO NOT INFER that they felt like a scientist merely from ROUTINE lab activities (running a procedure, collecting data, using equipment); however, scientist-like INTELLECTUAL engagement in the student's own voice - forming and testing their own ideas or hypotheses, building models, reasoning about how theory matches real-world behavior, or exploring physics more deeply than before - CAN support FLS even without the literal word "scientist"
        - M2: Student explicitly discusses feeling that the skills they learned are applicable to their future or their work as a scientist/researcher/engineer; DO NOT INFER that the skills will be applicable to their future work based on the activities they did in lab

        INCLUDE WHEN:
        - I1: Student says they felt like a scientist, experimental scientist, researcher, or physicist while doing the work.
        - I2: Student discusses feeling that the skills they learned are applicable to their future or their work as a scientist/researcher/engineer.
        - I3: Student frames themselves in the role of a scientist/researcher/engineer (e.g., "thinking like a scientist", "working like a researcher").
        - I4: Student associates science as the work of human researchers or scientists and references understanding what the researchers themselves do; learning how the EXPLICITLY INCLUDED NOUN researcher/scientist/physicist does science counts but must include the identity noun. This also covers advice-form sentences that frame a skill or technique as belonging to scientist/researcher practice — e.g., "the statistical tools scientists rely on in experiments" identifies the skill as what scientists do, associating the student's learning with scientist practice even without a first-person identity claim. 
        - I5: Student is giving advice in the form of "as a researcher..." or "as a scientist..." where the advice content itself reflects something only someone inhabiting that identity would say (a mindset, judgment call, or felt experience tied to the role) — not merely procedural instructions (measurement consistency, documentation, patience with equipment) that happen to be prefaced with the identity phrase.
        - I6: Student projects a scientist/researcher/experimentalist identity onto future students in advice form — e.g., "grow as a scientist", "develop as an experimental scientist", "their growth as an experimentalist", "you will become a better researcher", "learn from mistakes as an experimental scientist". The student is signaling that the class cultivates this identity even without claiming it for themselves directly. NOTE: the identity noun (scientist/researcher/physicist/experimentalist) must be present AND used as a substantive identity/role claim, not merely as the salutation or audience label for the memo ("Dear Future Researcher," "future researchers should...", "for new students/researchers" as an address form does NOT by itself satisfy I6). Generic advice about skills, improvement, or course mechanics without a substantive identity claim does not qualify, even when addressed to "future researchers." IMPORTANT DISTINCTION: a predicate sentence that explicitly assigns researcher/scientist identity — e.g., "now that you're a researcher" or "you are now a scientist" — IS a substantive identity projection (the student is asserting that taking the class confers that identity); this is NOT a mere salutation. Distinguish: "Dear Future Researcher" (address form, no identity assertion) vs. "now that you're a researcher, here is some advice" (identity claim that the class makes you a researcher).

        EXCLUDE WHEN:
        - X1: The student says the work they did *was* real research without expressing an identity/feeling dimension (that is DRR — Did Real Research: asserting the work itself was real or authentic, not just a feeling of identity).
        - X2: The student talks about *learning about* how research/science/experimental physics works without expressing an identity dimension (that is URR — Understanding Real Research: gaining conceptual understanding of research/science without expressing a researcher identity) EXCEPTION: Student says they learned how the NOUN researcher/scientist/physicist does science or thinks about, in which case they are relating their experience to that of a real scientist and it still counts as FLS 
        - X3: The student is describing their thinking or knowledge gained without an explicit identity component OR that the knowledge is exlicitly about how researchers/scientists/physicists operate OR explicitly linking the knowledge to future careers or research outside the classroom OR 
        - X4: Student says they "learned how to" do something or "learned how" research works UNLESS they are learning how the NOUN researcher/scientist/physicist does science, in which case they are relating their experience to that of a real scientist
        - X5: Exclude descriptions of tangible experimental skills or knowledge gains without explicit identity framing
        - X6: The future applicability of the skills learned in the lab is limited to more school contexts (e.g., other labs and projects)
        - X7: Span focuses on autonomy, freedom, or independence in the project (choosing topics, choosing methods, designing experiments) without an explicit scientist/researcher identity claim or without naming future research/lab work as the context — that is S7 (Decision Making: student autonomy, choice, or agency in designing or conducting their experiment), not FLS.

        EXAMPLES (Positive):
        - "This class was a solid first look at how scientists think through and analyze experiments."
        - "I hope this lab helps you grow as a researcher the way it helped me."
        - "Their aim should be to learn something, whether the hypothesis holds up or surprises push them to grow, picking up lessons from mistakes as an experimental scientist would." (advice-form identity projection — "as an experimental scientist" signals the student believes the class cultivates that identity)
        - "Get excited, because in this class you see how researchers come up with new ideas about how the world works."
        - "What you learn here is genuinely useful, particularly if you plan to do research later on."
        - "The most useful thing you'll pick up here is the statistical tools scientists rely on in experiments, like quantifying uncertainty and running chi-square tests." ✓ — "the statistical tools scientists rely on" frames the skill as belonging to scientist practice (I4); qualifies even without a first-person identity claim.
        - "Now that you're officially a 'researcher,' here are a few pieces of advice as you get started." ✓ — "you're now a researcher" is a predicate identity projection (I6), not a salutation; the student is asserting that the class confers researcher identity.

        EXAMPLES (Negative):
        - "Compared with a typical intro lab, this one is much more about real-world physics and actual data." (About the work being real → DRR, Did Real Research)
        - "This lab was engaging, and I could feel my understanding of experimental physics improving." (About understanding the discipline broadly → URR, Understanding Real Research)
        - "If you keep practicing the skills you have and the ones you still need, you'll be more successful in whatever you do next." (Not clear that the future endeavors are research/science related)

        SPAN GUIDANCE:
        - Select spans that include enough surrounding context to show the identity or feeling dimension.
        - Avoid selecting isolated phrases like "scientist" or "researcher" without the rest of the sentence.
        - Prefer full sentences even if the sentence contains some extra irrelevant material.

        CO-OCCURRENCE POLICY:
        - If a span meets the criteria for this code, assign it even if it may also qualify for other codes.
        - Do not attempt to resolve overlap.
    """).strip(),

    "URR": textwrap.dedent("""
        URR (Understanding real research)

        MUST INCLUDE:
        - M1: Clear explicit evidence in the words of the student that they gained a better understanding of what science, research, or experimental physics is like as a discipline or process (not merely as a list of class tasks). The student does NOT need to explicitly say "outside the classroom" — disciplinary or research-process language carries the broader frame on its own (see I7/I8).
        - M2: Students MUST strongly indicate or state explicitly that THEY BELIEVE they learned about how the process of science, research, or experimental physics works; DO NOT INFER that they understand real research based on narratives of activities they did in the lab

        INCLUDE WHEN:
        - I1: Student describes gaining an understanding of what it is like to do research or what research involves, or commentary about the nature of science or physics more broadly
        - I2: Student says they learned about or understand "experimental physics" or "experimental science" referring to the discipline broadly
        - I3: Student says they have a better understanding of conducting experiments in a broader research or scientific context than the classroom. Includes the broad understanding of "experimenting" or "experimentation", especially when combined with terms like "actual" or "real"
        - I4: Student uses phrasing like they "learned how to" do science/experimental physics/research — the "learn" differentiates it from DRR (Did real research) which is more active This INCLUDE applies only when the object of learning is the discipline or research PROCESS itself (experimental physics, how research or experimentation works, developing hypotheses, research methods); a specific lab procedure or technique (conducting or designing a particular experiment, error analysis, coding, data analysis) is S6, not URR (see EXCLUDE WHEN).
        - I5: PROCESS is key. Emphasizes elements of the research process or features of science/physics.  Can include "learning how to do science/research/physics" as well as "learning how real science/research/physics is done"
        - I6: There is collective evidence in the sentence by use of terms like science, physics research, experimental physics, experimentation broadly, experimenting, or a combination of these kinds of words that the sentiment is broader than the context of the class   
        - I7: "Experimental physics," "experimental science," "experimentation," "research techniques," and "methods of research" are inherently disciplinary terms — when a student uses them in a learning, understanding, OR teaching frame ("I have a better understanding of experimental physics," "this course teaches you all about experimental physics," "I learned research techniques"), this satisfies the broader-than-classroom requirement on its own. Do not additionally require phrases like "outside the classroom" or "in the real world" when these disciplinary terms are present. MANDATORY ACCEPT: if any of these disciplinary anchors appear in a learning/understanding/teaching context, accept the span — do not downgrade based on surrounding skill lists or that the phrasing feels promotional.
        - I8: A claim that the student learned about or gained understanding of experimental physics, the research process, experimenting, developing hypotheses, or research methods QUALIFIES as URR on its own - accept it; do not park it in review waiting for an explicit "I now understand how research works" meta-statement. The disciplinary or process language IS the understanding claim.
        - I9: Explicit declarative statements about the nature of research or experimental physics count even without "I learned..." framing — e.g., "Real research is messy and rarely goes the way you expect" or "Science is all about iteration and learning from failure." The student is conveying an understanding of what research is like; the absence of a learning verb does not disqualify the span.
        - I10: Learning about or understanding "actual" science, research, or experimenting is a strong URR signal — students use it to distinguish genuine research from typical classroom work; CAUTION: this can also be a signal for DRR (Did real research) when students say the class felt like an actual research experience or that they collected actual data. The difference is whether it was learning about the actual research or taking part in the actual research.
        - I11: Advice-form sentences qualify when: (a) the advice states a disciplinary or process-level characterization of how research/science actually works (e.g., "research is messy and rarely goes as planned," "science is about iteration and learning from failure," "real research means questioning your own conclusions"); OR (b) the advice characterizes the course as being about "experimental physics," "research techniques," or "methods of research" as a discipline (e.g., "this course teaches you all about experimental physics" — per I7, the disciplinary anchor qualifies the span even in advice/course-description form). Do NOT count ordinary procedural/practical tips dressed in advice form — reminders to be consistent, patient, careful with measurements, not stressed about uncertainty, or to document/log things properly are S6/S7 territory even when addressed to "future students" or "future researchers," UNLESS the tip explicitly frames the underlying reason as a fact about the discipline/process itself, not just a study habit. "Future researchers/students" as a salutation or audience marker is not itself evidence of URR — the *content* of the advice must be the disciplinary claim, not the framing.
        - I12: The student references the guest lectures by people at the university who talked about their research and discuss it giving them a better idea of the research going on at the school or how the class relates to the guest lecturer's research

        EXCLUDE WHEN:
        - X1: The student says the work they did *was* real research, or that the class simulated / resembled / felt like actual research, or uses "real-world"/"real-life" to describe their own experiment or class — that is DRR (claiming the work was authentic/genuine), not URR. URR requires the learning/understanding framing, not a claim that the work itself was real.
        - X2: The student expresses an identity-based feeling of being a scientist or becoming a better experimentalist/researcher — that is FLS. STILL CODE AS URR if any part of the phrasing is about learning or understanding what researchers/physicists/scientists do or what experimental physics/science is about. Mere identity adornment ("as an experimental scientist," "as a future researcher") attached to advice or activity descriptions does NOT by itself satisfy the URR exception — there must be a substantive learning claim about the discipline or research process.
        - X3: The student lists or names SPECIFIC, NAMED lab procedures or techniques (e.g., "coding," "graphing," "statistical testing," "using Colab," "quantifying error") AND the span contains NO reference to the discipline or research process — that bare-list case is S6, not URR. CRUCIAL: a reference to the discipline itself — "experimental physics," "scientific experiments," "experimentation," "research techniques," "methods of research," "research" as a field, or to the research PROCESS — STILL COUNTS as URR EVEN WHEN it is embedded in or surrounded by a list of specific skills. The presence of a skills list does NOT downgrade a span that also names the discipline. Accept e.g. "develop your experimental physics knowledge," "learn to correctly design and run scientific experiments," "showed me a lot about experimental physics, especially collecting and analyzing data," "I learned a great deal about experimental physics, especially gathering and analyzing data, estimating uncertainty, and designing sound experiments," "This class introduced me to research techniques like collecting and analyzing data," "I learned about both physics principles and methods of research," "I came to understand the overall process of experimenting, why experiments matter, and how they are applied," "I learned how digital tools can push research forward" — the disciplinary term qualifies the whole span regardless of the procedures or tools listed alongside it. X3 applies ONLY when the span is purely bare named procedures with NO disciplinary/process reference anywhere in it.
        - X4: Span describes freedom, independence, or student agency in designing or conducting experiments, with no reference to research/experimental physics/physics-as-a-discipline as the broader frame — that is S7, not URR.
        - X5: The student says they understand physics content or theory more deeply ("I finally understand how Hooke's Law works," "this course deepened my understanding of electromagnetism") — that is content learning, not URR. EXCEPTION: if the framing is about how theory connects to experimental practice ("I expected physics to be all theory, but this lab showed me how the equations hold up in real experiments"), that IS URR.
        - X6: Span is purely class logistics or grading commentary (workload, deadlines, grading style, lab-report format) with no characterization of the scientific/research process — not URR. NOTE: a description of the course as being ABOUT the scientific or research process (e.g., "the class is more about the process of running good experiments than about the concepts") DOES qualify as URR.
        - X7: Pure first-person narration of the student's OWN specific experiment, results, or research question — e.g., "our research question was how X correlates with Y," "we measured the muon count and found...," "our group tested whether..." — is NOT URR; describing what you specifically did or found is narration/DRR territory, not an understanding-of-research claim. EXCEPTION: it IS URR when the narration is used to draw a generalization about how research/science actually works (e.g., "...which taught me real experiments rarely go as planned").

        EXAMPLES (Positive):
        - "This lab taught me a great deal about experimental physics and the tools you need to run experiments." (explicit combination of "experimental physics" indicates knowledge about science more broadly)
        - "The class was hard but worth it; beyond methods and data analysis, I learned a lot about teamwork, communication, and solving problems in a research context." ("in a research context" illustrates applicability of the skills beyond class to a research setting)
        - "Honestly, this class changed the way I think about research." (shows changed understanding of the broader research endeavor)
        - "I now have a much better understanding of experimental physics." ✓ — "experimental physics" refers to the discipline broadly; no additional "outside the classroom" qualifier is needed (I7).
        - "The focus is less on physics concepts and more on the scientific process of running good experiments." — course-process description: the student conveys understanding of what experimental work centers on, even framed as course commentary.
        - "This class introduced me to several research techniques, including data collection, uncertainty, distinguishability, and analysis." ✓ — "research techniques" is a disciplinary anchor (I7/X3); the named techniques alongside it do not disqualify.
        - "I learned a great deal about experimental physics, especially gathering and analyzing data, estimating uncertainty, and designing sound experiments." ✓ — "experimental physics" is the object of learning; the skill list is explicitly accepted per X3 and I7.
        - "In this course, I came to understand the overall process of experimenting, why experiments matter, and how they are applied." ✓ — "overall process of experimenting" and "why experiments matter" are process-level characterizations (I3, I5).

        EXAMPLES (Negative):
        - "What I remember most wasn’t the physics itself; it was working as a team to answer an actual research question and fix problems along the way." (Even though it says process, indicates they actually did the work being real → DRR)
        - "Your goal here should be to learn to think like a researcher and bring that mindset to your final project" (Primarily about learning to think like a scientist → FLS)
        - "This semester I learned to run an experiment and to analyze and interpret the results." (Not clear the student is relating to science/research more broadly)
        - "The class gave me useful practice in experimental physics, analyzing data, and communicating science." (List of class deliverables; "experimental physics" appears but is a topic in a list, not the object of an understanding claim → DRR/S6, not URR)

        SPAN GUIDANCE:
        - Select spans that include enough surrounding context to show what the student now understands about research.
        - Avoid selecting isolated phrases like "understanding" or "experimental physics" without the rest of the sentence.
        - Prefer full sentences even if the sentence contains some extra irrelevant material.

        CO-OCCURRENCE POLICY:
        - If a span meets the criteria for this code, assign it even if it may also qualify for other codes.
        - Do not attempt to resolve overlap.
    """).strip(),


    "S3": textwrap.dedent("""
        S3 (Failure)

        MUST INCLUDE (at least one of):
        - M1: Difficulties experienced during the lab such as things going wrong, struggling to get something to work, or major issues encountered that forced students to redo work or change course
        - M2: GENERAL OR BROAD observations about the non-ideal or messy nature of data/science/experimental work (mentions of mistakes, problems, or implied failure like troubleshooting, improving, fixing, or adapting count)
        - M3: Student observes a need for patience or experiments tending to take longer than expected
        - M4: Failure implied through words like "fix", "debug", "troubleshoot", etc. (this can also be what failure felt like: "uncomfortable", "confusing", "hard", "difficult", etc.) or failure made explicit through words like "mistake", "struggle", "challenge", "problem", etc.

        INCLUDE WHEN:
        - I1: Student uses words like "trial and error", "troubleshoot", or "debug" to describe their lab experience
        - I2: Students describe things being generally difficult or challenging, or have issues with getting an experiment to work, or have issues with analysing the data
        - I3: Student notes that data is "messy", uncertain, not ideal, or difficult to fit well
        - I4: Student discusses something that seemed like a rather large catastrophe for their group that likely caused major distress

        EXCLUDE WHEN:
        - X1: The student is talking about the class being bad or that they did not learn anything
        - X2: Discussion of one group's actual results, which must be explicitly mentioned within the span, not coming out as they expected--this does not include broad statements about a group's results or results in general being unexpected sometimes, which is S3.
        - X3: Talking about struggles with Google Colab exclusively. If a span mentions struggling with code, people, the experiment, etc. this is S3. However, just saying "I hate Colab" is not.
        - X4: Discussing experimental error
        - X5: Failure is exclusively due to not having enough time in class to collect data

        EXAMPLES (Positive):
        - "It was frustrating early on, but learning to make sense of messy, imperfect data ended up being one of the best parts of the class."
        - "That said, our experiment wasn't perfect; it had some flaws and errors."
        - "One big lesson we learned: a hypothesis that turns out wrong doesn't mean the experiment failed."

        EXAMPLES (Negative):
        - "Get used to Colab early. It's really useful, but it can be frustrating if you're not familiar with it." (About frustration with Google Colab)
        - "It's really important to learn to spot sources of error and reduce them." (About improving experiment → more likely S5 (Iteration: refining or improving experiments/methods))
        - "We only had time to collect a small dataset, so measuring many more stars would probably
        show a clearer trend.” (About time in class to collect data)

        SPAN GUIDANCE:
        - Select spans that include enough surrounding context to show what aspect of their experience the student associated with failure
        - Avoid selecting isolated phrases like “fail” or "troubleshoot" without the rest of the sentence.
        - Prefer full sentences even if the sentence contains some extra irrelevant material.

        CO-OCCURRENCE POLICY:
        - If a span meets the criteria for this code, assign it even if it may also qualify for other codes.
        - Do not attempt to resolve overlap.
    """).strip(),


    "S5": textwrap.dedent("""
        S5 (Iteration)

        MUST INCLUDE (at least one of):
        - M1: Statements about reducing error in, refining, or improving data/uncertainty/experiments in general
        - M2: How repeating certain activities in their experiment improved it
        - M3: Changing approach to their experiment or doing another experiment built on what they had started
        - M4: General observations that iteration is important in experimentation

        INCLUDE WHEN:
        - I1: Terms like "repetition", "iteration", "redo", "refine", "improve", “trial and error”, "adjust", "troubleshoot", "recalibrate", "modify", "rebuild", "adapt", "redesign", etc. are used, though more are possible
        - I2: Student described learning about the importance of making improvements and/or repeated trials
        - I3: Discussion of activities done to reduce uncertainty or minimize error
        - I4: Student talks about designing a new experiment after a different one failed
        - I5: Student describes learning from a mistake or failure and applying that insight to improve their approach
        - I6: If in the specific context of a student's experiment, must indicate that improvement is being made to the experiment, not just that multiple trials were conducted
        - I7: Describing learning or being taught experimental iteration
        - I8: Presenting future student or advice-form framing of broad (NOT this-would-only-work-within-one-experiment-specific) iteration (e.g., "a new student should consider factors and try to reduce them", "be creative in finding new ways to improve precision")

        EXCLUDE WHEN:
        - X1: The student talks about taking multiple trials but does not mention or imply reducing uncertainty, getting "better", "reliable", etc. data/results, or arriving at some conclusion at all
        - X2: Students are just making specific, within-their-own-experiment-level recommentations to future students about how THEY could iterate/improve on the work. Describing iteration in general to future students is S5.

        EXAMPLES (Positive):
        - "In each lab we designed our own procedure, wrote it up carefully, and then improved it based on what didn't work."
        - "Getting this setup working was hard and took us several rounds of changes."
        - "The class emphasizes designing experiments, trial and error, and making sense of uncertainty."

        EXAMPLES (Negative):
        - "New students should think early about how to extend or refine their project and set clear goals." (Advice to future researchers)
        - "We reached this conclusion by taking lots of measurements, graphing them, and running a chi-squared test." (Not clear many measurements is linked to refinement)
        - "The project could be extended with more data and more analysis of the fits." (Advice to future researchers)

        SPAN GUIDANCE:
        - Select spans that include enough surrounding context to show the student did iteration to refine their experiment
        - Avoid selecting isolated phrases like “refine” or "iteration" without the rest of the sentence.
        - Prefer full sentences even if the sentence contains some extra irrelevant material.

        CO-OCCURRENCE POLICY:
        - If a span meets the criteria for this code, assign it even if it may also qualify for other codes.
        - Do not attempt to resolve overlap.
    """).strip(),

    "S6": textwrap.dedent("""
        S6 (Scientific practices)

        Scientific practices include teamwork, coding, experimenting, data analysis, experimental design, critical thinking, and presenting work. If any of these are mentioned--even vaguely or within the context of one experiment without any further reflection--they should still be coded as S6.
        Please consider the warning above X8 and X9 before applying either code.

        MUST INCLUDE:
        - M1: Statements indicating that the student DID, LEARNED TO DO, OR NAMES specific scientific practices in the context of THIS class or their final project. Saying they did a certain scientific practice counts as S6, even if they hadn't learned it or reflected on it deeply.
        - M2: First- or second-person singular/plural framing, or third person framing related to the lab/class ("I learned…", "You will use…", "we calculated/fit/analyzed…", "we measured/tested…", "This lab helps develop skills in…") naming a practice or skill is sufficient — explicit reflection or evaluation language is NOT required
        - M3: Bare topic descriptions ("our project investigated X") still do NOT qualify, but listing, naming, or directly implying the practices used does

        INCLUDE WHEN:
        - I1: There is any mention of designing experiments
        - I2: The student REFLECTS on or SIMPLY NAMES/IMPLIES specific scientific practices they engaged in or learned as part of their experience. Some scientific practices are: coding, data analysis (statistical testing, graphing, etc.), teamwork, experimental design, experimentation, critical thinking, topic selection, and presenting work.
        - I3: The student discusses scientific practices used in their project with reflective or evaluative framing — not merely narrating experimental steps
        - I4: Using Google Colab as it relates to learning how to code or analyze data, plot, keep notes and/or keep data organized
        - I5: Providing generic advice to future students about scientific practices they will learn or broadly reflecting on scientific practices the student already learned
        - I6: Listing experimental steps!!!
        - I7: The student notes a term related to a scientific practice by saying that the course (PHYS 1110) taught them this skill or that they learned it in lecture/lab

        EXCLUDE WHEN:
        - X2: Rote statement of results that don't link to an explicit action like data analysis/graphing/fitting/calculating/measurement (mentioning that statistical tests/graphs/fits contributed to a conclusion implies that they were made/done, which is a scientific practice) (NOT S6 e.g., "At steep angles the balls rolled quickly regardless, so the ramp angle no longer seemed to affect the spin." IS S6: "Based on the t' test, our hypothesis was not supported.")
        - X5: Exclusively describing working efficiently/pacing your work without any mention of other scientific practices (data collection, keeping lab notebooks, coding, analysis, plotting, etc.) does not count
        - X6: Mentioning Google Colab EXCLUSIVELY. Needs to mention specific tasks they did with Colab (e.g., coding, analysis, plotting, keeping a lab notebook etc.). Other tools like Python or experimental devices can be mentioned without noting specific tasks, as they directly imply coding and experimentation, respectively, while Colab does not.
        - X7: The skills were exclusively learned in a time other than this semester and in a place other than THIS LAB, THE TASKS INVOLVED WITHIN IT, OR LECTURE (there is a lecture corresponding to lab). First-person framing is not required for a span to be S6. Also, comparing gaining skills in this class to gaining skills elsewhere still counts as S6: as long as students mention gaining skills in this class at all, the span is S6.
        - X8: The statement is NOT a narrative walkthrough of the experiment, but PURELY a project TOPIC or RESULT description (e.g., "our project investigated X").
        - X9: COMPLETELY SPECIFIC advice about one tool/technique. Broad advice can count as S6. (NOT S6 e.g., "next time, try filming it instead," "use a sturdier ramp," "have something ready to catch the robots").


        EXAMPLES (Positive):
        - "The skills I built most in this class were communication, teamwork, and persistence."
        - "To answer it, we collected and cleaned our data and then fit several models in Colab."
        - "The main things I learned were how to design experiments and how to analyze what they produce."

        EXAMPLES (Negative):
        - "For our final project we looked at how ambient light changes the noise in our detector." (Just a description of the lab, not a specific task)
        - "After comparing chi-squared values across several sets of trials, we concluded amplitude varies linearly with time." (Discussing results of the experiment not tasks they did)
        - "Colab was easier to use than I expected, but it's worth spending some time getting used to it." (Not indicating a specific task performed with Colab)
        - "Our team's final project looked at how different surfaces change energy lost to friction." (Project topic description — no reflection on scientific practices engaged in)

        SPAN GUIDANCE:
        - Select spans that include enough surrounding context to show the specific scientific practices students engaged in
        - Avoid selecting isolated phrases like "data collection", "coding", "teamwork", etc.) without the rest of the sentence.
        - Prefer full sentences even if the sentence contains some extra irrelevant material.
        - This code is more common than the others and will generally be applied most often

        CO-OCCURRENCE POLICY:
        - If a span meets the criteria for this code, assign it even if it may also qualify for other codes.
        - Do not attempt to resolve overlap.
    """).strip(),

    "S7": textwrap.dedent("""
        S7 (Decision making)

        This code is not just about agency, freedom or creativity, though these are all important aspects of decision making.
        0) Mentions of creativity, agency, or freedom that *directly lead to decisions being made, which the student mentions or implies* should be coded.
        Specific, in-procedure decisions should most often not be coded as S7 because they generally do not imply broad decision-making ability.
        Ex: freedom in designing one's own experiment is S7, but testing a certain variables is not S7 (specific in-procedure decision rather than broader freedom to choose the direction of work)

        For context, as part of this class, students choose the topic of their final project. Mentions of the final project alone are not S7, but references to how students made decisions about this project could be S7.
        Also, part of the prompt for these memos include providing descriptions of how to continue an experiment (questions, procedures, data analysis techniques, etc.). Most of these descriptions will not be S7 because they are specific to one experiment and NOT indicative of overall decision making opportunity, but take a look.

        MUST INCLUDE (at least one of):
        M1 - Indication that students felt independent or able to make their own decisions in the lab
        M2 - Indication that students could get creative or curious about their approaches/solutions

        INCLUDE WHEN:
        I1 - There is a sense of agency associated with the statement that directly implies a decision broader than the individual experiment being made. The combination of 3 + 4 alone is not enough to be S7
        I2 - Students indicate they had autonomy of choice!! (e.g., choosing a topic, deciding how to answer a research question) *WITHOUT simply stating the specific choice that was made*. Specific experimental decisions do not imply broad freedom to choose the direction of work and should not count as S7. EXCEPTION: when the *act of choosing itself* is the subject — explicit deliberation over which method, approach, or variable to pursue (e.g., \"there are many choices to be made, so choose a method that...\", \"decide which variable to experiment with\") — it IS S7, because the student owns that decision. This is distinct from merely naming the variable that was tested (see X6).
        I3 - Students note different factors or constraints (implicit or explicit) that influenced an implicit or explicit decision they made
        I4 - Students express that they were able to follow their interests in the lab
        I5 - Students say that they were able to exercise creativity in their work (creativity alone is not sufficient, the ability to express that creativity is important)
        I6 - (IMPORTANT) Students describe more specific choices in one experiment when combined with a reference to broader autonomy/decision making in the immediate surrounding text
        I7 - Advice to future students that *EXPLICITLY, NOT implicitly* describes the autonomy, creativity, or choice they will have in the lab/project without solely stating decisions made or to be made within the scope of their individual experiment (Good e.g., \"you'll come up with your own experiment\", \"you'll be able to choose your topic\", \"you'll have the freedom to explore something you're curious about\", \"you can try obscure things\") — advice from S6 does NOT preclude S7 when the substance is about agency/autonomy. (Bad e.g. \"you could try testing how height affects the outcome or vary the type of string\")
        I8 - Terms such as "decide", "create/creative", "choose", "freedom", "own", "explore", "curious", any of these words in noun/verb/adjective form, or other words that imply autonomy, creativity, or decision making are used

        DO NOT REJECT THESE — they ARE S7 (judge by SUBSTANCE, not by whether agency words appear):
        - PROJECT-level decisions: choosing/settling the topic, research question, goal, scope, direction, or analysis approach — including advice to do so (e.g., "settle on a topic to research", "think about what you're trying to learn or prove", "expand a prior lab into your own project", "keep the project manageable", "think outside the box to find new things to try"). A span does NOT need the words choose/decide/freedom/own to count — an implied project-level decision is enough.
        - TEAM-level decisions: choosing group roles, partners, or how to delegate (e.g., "choose suitable roles in your group", "be open to who you work with", "how to delegate tasks").
        - OWN-EXPERIMENT / OPEN-INVESTIGATION autonomy: coming up with or designing your own experiment, or being able to test/investigate whatever you want (e.g., "come up with a unique experiment", "test anything you wanted", "investigate whatever interests you"). Do NOT dismiss these as "just a course requirement" or as a "specific in-procedure variable choice" — the freedom to design/test what you want IS S7, even stated as a requirement.
        - INDIVIDUAL intellectual agency / voice: advocating for your own ideas, sharing your voice, or choosing to find answers yourself instead of taking things at face value (e.g., "advocate for yourself and your ideas", "share your voice", "search for answers yourself").
        These are NOT X4. X4 covers ONLY within-a-single-procedure knob-turning (more trials, slow-mo camera, different surface). Topic / goal / scope / direction / team decisions are never X4.

        EXCLUDE WHEN:
        X1 - The students are making judgements or interpretations about their data/results
        X2 - Describing specific experimental thought, design, decisions within the scope of one procedure IF AND ONLY IF there is absolutely no implication at all of broad (experiment-level or beyond) decision making or autonomy in the lab (ex: listing methods is not S7, but saying that you could coose them on your own or set your own broader goals for the project is.) (e.g., NOT S7: \"we chose to do 3 trials instead of 4\", S7: \"since the force sensor gave bad readings, we hung known masses and computed the spring force from gravity instead\")
        X3 - Saying *only* that they learned to design an experiment does not count (has to be qualified with another phrase such as \"my own experiment\" or \"design an experiment from scratch\").
        X4 - (IMPORTANT) Giving advice to future students exclusively about specific iteration MECHANICS (e.g., \"do more trials\", \"use a slow-motion camera\", \"try a different surface\") — this is distinct from advice about future students' autonomy/creativity (eg. choosing group mates, project directions, etc.), which IS S7
        X5 - The span ONLY describes practices/skills the student used or developed/learned (S6 — Scientific Practices: skills or techniques the student learned or used) with NO wording suggesting or implying choice, ownership, autonomy, creativity, or freedom. Wording like \"got to\", \"able to\", \"had the freedom to\", \"my own\", \"from scratch\", \"of our choosing\", \"I picked\", \"I decided\", \"you can choose\" all tip a span INTO S7 (possibly co-occurring with S6). Independence, by the way, is not a skill!
        5b - If the span falls under X5, see I8, which overrides it.
        X6 - Describing specific variables to change/test (not variables in general, which can be S7--specific ones are not) or the design of an experiment in the future without any reference to or implication of broader decision making in the experiment's design/direction or the lab class overall or reasoning behind that decision! NOTE: the exclusion is for *naming the variable tested* (\"we varied string type\", \"test how height affects the outcome\"). Explicit choice/deliberation over WHICH variable or method to use (\"many choices to be made\", \"decide a variable to test\") is I2, not X6.

        EXAMPLES (Positive):
        - \"Future researchers should design their method carefully; there are many choices to be made, and you want the one that gives the least uncertainty.\"
        - \"New students should aim to design an experiment they actually care about, since that keeps them more engaged with it.\"
        - \"At first I was nervous about having to come up with my own ideas and procedure.\"
        - \"This lab showed me how fun experimenting can be when you get to be as creative as you want.\"

        EXAMPLES (Negative):
        - \"So we set out to test how surrounding light levels affect detector noise, to understand its source and how the detector design might be improved.\" (Not indicative of broad autonomy)
    """).strip(),
}

def V_AGENT_SYSTEM(s_code: str) -> str:
    _ctx_num = 4 if USE_EXEMPLAR_BANK else 3
    _context_item = (
        f"{_ctx_num}. Surrounding context — the candidate span is wrapped in [FLAGGED] ... [/FLAGGED] markers;\n"
        "   preceding and following sentences appear as plain text for context only.\n"
        "   Base your decision on the flagged portion; use surrounding text only to resolve\n"
        "   ambiguity (e.g. negation, coreference, or framing that changes the meaning of the flagged span)."
        if CONTEXT_SENTENCES > 0 else ""
    )
    _context_line = f"\n{_context_item}" if _context_item else ""
    if USE_EXEMPLAR_BANK:
        _bank_block = (
            f"3. Retrieved exemplars from three sources — each shows the correct code(s) and reasoning:\n"
            f"   a) Direct {s_code} exemplars — Typical, Close but yes, and Close-but-no from the {s_code} bank.\n"
            f"   b) Co-occurrence exemplars — spans from other codes also legitimately co-coded as {s_code} (positive signal).\n"
            f"   c) Cross-code boundary exemplars — spans from other codes where {s_code} appears as a boundary reference:\n"
            f"      - Close but yes for another code with {s_code} as confusion risk \u2192 span IS that other code, NOT {s_code} (reject).\n"
            f"      - Close-but-no for another code with {s_code} as correct code \u2192 span IS {s_code} (keep)."
            + _context_line
            + "\n\nExemplar signal guide:\n"
            f"- Direct Typical for {s_code}: strong positive signal \u2014 keep if similar.\n"
            f"- Direct Close but yes for {s_code}: confirmed positive (non-prototypical) \u2014 keep if similar; rely on explanation.\n"
            f"- Direct Close-but-no for {s_code}: looks like {s_code} but wrong \u2014 reject if similar.\n"
            f"- Co-occurrence with {s_code}: span IS both another code AND {s_code} \u2014 positive signal.\n"
            f"- Cross-code Close but yes ({s_code} = confusion risk): span IS another code, NOT {s_code} \u2014 reject.\n"
            f"- Cross-code Close-but-no ({s_code} = correct code): span IS {s_code} \u2014 keep."
        )
    else:
        _bank_block = _context_line.lstrip("\n") if _context_line else ""
    return f"""
You are a precision-filter specialist for the {s_code} code in a two-stage qualitative coding pipeline
for student physics lab memos.

The span-finder (first stage) was intentionally set to cast a wide net — it flags any span that
plausibly matches {s_code}, including borderline and ambiguous cases. Your job is to act as the
precision filter: critically evaluate each candidate span and decide whether it genuinely meets
the criteria for {s_code}. Expect a meaningful number of false positives — that is fine. A human reviewer will checks every span you keep, so a wrongly-KEPT span is cheap to discard while a wrongly-REJECTED span is lost for good. Therefore LEAN TOWARD ACCEPTANCE: keep any span that plausibly fits, and reject only a span that clearly fails the criteria or clearly belongs to a different code.

Other codes exist in this codebook, but your ONLY task is to judge whether this span meets the
criteria for {s_code}. If a span clearly belongs to a different code, that is still a rejection —
do not attempt to identify what other code applies.

You will be given:
1. The candidate span — text flagged as potentially matching {s_code}.
2. The full {s_code} code definition (INCLUDE WHEN, EXCLUDE WHEN, examples).
{_bank_block}

Guidelines:
- The code definition is the primary authority. Exemplars are decision aids, not substitutes — the bank is not
  exhaustive. A span may legitimately qualify for {s_code} even if no retrieved exemplar closely resembles it.
- Calibrate how much weight to give each exemplar by how well it aligns with the candidate span:
    • High similarity → strong signal in that exemplar's direction (positive for Typical/Close but yes/co-occurrence;
      negative for Close-but-no and cross-code Close but yes confusion-risk exemplars)
    • Low similarity → weak signal; do not let a poor-fit exemplar drive the decision
- When exemplars are strong matches: let them reinforce or challenge how you read the definition.
- When exemplars are uniformly weak matches: fall back to the code definition directly. Ask — does this span
  meet the INCLUDE WHEN criteria and avoid the EXCLUDE WHEN criteria? That answer should drive the decision.
- The EXCLUDE WHEN section defines hard boundaries — apply it strictly regardless of exemplar evidence.
- When a span only partially or ambiguously meets the criteria, KEEP it — lean toward acceptance on every
  borderline call; reject only a clear mismatch or a span.

Return STRICT JSON ONLY:
{{
  "keep": true or false,
  "rationale": "<one sentence explaining your decision>"
}}

No commentary, no extra keys.
""".strip()



def V_AGENT_USER(s_code: str, span_text: str, retrieved_exemplars: list, code_def: str,
                 before_text: str = "", after_text: str = "") -> str:
    exemplar_lines = []
    for ex in retrieved_exemplars:
        pool  = ex.get("pool", "direct")
        pcode = ex["s_code"]
        etype = ex["exemplar_type"]
        text  = ex["exemplar_text"]
        expl  = ex.get("explanation", "").strip()
        bc    = ex.get("boundary_code", "").strip()
        coc   = ex.get("co_code", "").strip()

        if pool == "direct":
            line = f"  [{pcode} | {etype}] \"{text}\""
            line += f"\n    Correct code: {pcode}"
            if expl:
                line += f"\n    Why: {expl}"
            if bc:
                if etype == "Close but yes":
                    line += f"\n    Confusion risk: {bc} (wrong — this span IS {pcode})"
                else:
                    line += f"\n    Correct code is actually: {bc} (NOT {pcode})"
            if coc:
                line += f"\n    Also coded: {coc} (legitimate co-occurrence)"

        elif pool == "co_code":
            line = f"  [{pcode} + {s_code} co-occurrence | {etype}] \"{text}\""
            line += f"\n    Correct codes: {pcode} AND {s_code} (this span legitimately has both)"
            if expl:
                line += f"\n    Why: {expl}"

        elif pool == "boundary":
            if etype == "Close but yes":
                line = f"  [{pcode} Close but yes — {s_code} is confusion risk | {etype}] \"{text}\""
                line += f"\n    Correct code: {pcode}  |  {s_code} is WRONG here"
                if expl:
                    line += f"\n    Why {pcode} (not {s_code}): {expl}"
            else:
                line = f"  [{pcode} Close-but-no — correct code is {s_code} | {etype}] \"{text}\""
                line += f"\n    Correct code: {s_code}  (looks like {pcode}, but IS {s_code})"
                if expl:
                    line += f"\n    Why {s_code} (not {pcode}): {expl}"

        else:
            line = f"  [{pcode} | {etype}] \"{text}\""
            if expl:
                line += f"\n    Why: {expl}"

        exemplar_lines.append(line)

    exemplars_block = "\n".join(exemplar_lines) if exemplar_lines else "  (none retrieved)"
    if USE_EXEMPLAR_BANK:
        exemplar_section = (
            "\n\nRETRIEVED EXEMPLARS (top matches — direct, co-occurrence, and cross-code boundary signals):\n"
            + exemplars_block
        )
    else:
        exemplar_section = ""

    if before_text or after_text:
        context_note = " — surrounding sentences shown for context, flagged portion marked"
        parts = []
        if before_text:
            parts.append(before_text.strip())
        parts.append(f"[FLAGGED] {span_text} [/FLAGGED]")
        if after_text:
            parts.append(after_text.strip())
        span_display = "\n".join(parts)
    else:
        context_note = ""
        span_display = span_text

    return f"""
CANDIDATE SPAN (flagged as {s_code}){context_note}:
\"\"\"{span_display}\"\"\"

CODE DEFINITION for {s_code}:
{code_def}{exemplar_section}

Task:
Decide whether the candidate span should be kept as {s_code} or rejected. When the span is borderline or you are unsure, KEEP it (lean toward acceptance).
Return ONLY: {{"keep": true/false, "rationale": "<one sentence>"}}
""".strip()